In [ ]:
terms   = ['trained', 'untrained']
palette = dict(zip(terms, ['red', 'blue']))
offsets = np.linspace(-.6, .6, len(terms))          # dodge, in session units

fig, axs = plt.subplots(1, len(rois), sharey=True, sharex=True, figsize=(8, 2), constrained_layout=True)

for r, roi in enumerate(rois):
    ax = axs[r]
    for offset, term in zip(offsets, terms):
        t = lme[(lme.roi==roi) & (lme.term==term)].sort_values('session')
        ax.errorbar(t.session + offset, t.coef, yerr=t.se,
                    color=palette[term], lw=2, marker='o', label=term)
    ax.axhline(0, lw=.8, ls=':', color='k')
    ax.set_title(roi)
    ax.set_xticks(gl.sessions)

sb.despine(ax=axs[0], trim=True)
[sb.despine(ax=ax, left=True, bottom=True) for ax in axs[1:]]
axs[0].set_ylabel('dissimilarity (a.u.)')
axs[0].set_xlabel('session')
fig.legend(*axs[0].get_legend_handles_labels(), frameon=False, loc='lower right')

In [ ]:
rows = []
for roi, session in itertools.product(rois, gl.sessions):
    d = dist[(dist.roi==roi) & (dist.session==session)]
    model0 = smf.mixedlm('crossnobis ~ 0 + chord + crossnobis_group', d, groups=d['sn']).fit()
    model_diff = smf.mixedlm('crossnobis ~ crossnobis_group + chord', d, groups=d['sn']).fit()

    print(f'{roi}, session {session}, z={model_diff.tvalues["chord[T.untrained]"]}, p={model_diff.pvalues["chord[T.untrained]"]}')

    terms = model0.fe_params.index
    ci    = model0.conf_int()

    block = pd.DataFrame({'term':  terms,
                         'coef':  model0.fe_params[terms].to_numpy(),
                         'se':    model0.bse[terms].to_numpy(),
                         'z':     model0.tvalues[terms].to_numpy(),
                         'p_val': model0.pvalues[terms].to_numpy(),
                         'ci_lo': ci.loc[terms, 0].to_numpy(),
                         'ci_hi': ci.loc[terms, 1].to_numpy()})

    rows.append(block.assign(Hem=H, roi=roi, session=session, group_var=model0.cov_re.iloc[0, 0], converged=model0.converged))

lme = pd.concat(rows, ignore_index=True)
lme.term = lme.term.map({'chord[trained]': 'trained', 'chord[untrained]': 'untrained'})

In [ ]:
rows = []
for sn, roi, session, chord in itertools.product(gl.participants, rois, gl.sessions, ['trained', 'untrained']):
    data = dist[(dist.sn==sn) & (dist.roi==roi) & (dist.session==session) & (dist.chord==chord)][['crossnobis', 'crossnobis_group']].to_numpy()
    y    = data[:, 0]
    X    = np.c_[data[:, 1], np.ones(6)]
    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    rows.append(pd.DataFrame({'sn': sn, 'Hem': H, 'roi': roi, 'session': session, 'chord': chord, 'beta': beta[0], 'intercept': beta[1]}, index=[0]))

intercept      = pd.concat(rows)
intercept_tr   = intercept[intercept.chord=='trained']
intercept_untr = intercept[intercept.chord=='untrained']
intercept_diff = intercept_tr.merge(intercept_untr, on=['sn', 'Hem', 'roi', 'session'])

fig, axs = plt.subplots(1, len(rois), sharey=True, sharex=True, figsize=(8, 2), constrained_layout=True)

plot_im_sess(fig, axs, intercept, rois, y='intercept', x='session', hue='chord', hue_order=hue_order, palette=palette, kind='point', estimator='mean', dodge=.4, add_zero=True)
sb.despine(ax=axs[0], trim=True)
axs[0].set_ylabel('dissimilarity (a.u.)')
axs[-1].legend([], frameon=False, loc='upper right')
fig.legend(frameon=False, loc='lower right')
axs[0].set_xlabel('session')

plt.show()